# Chapter 18: Cost, Performance and Limits (notebook edition)

## Learning Objectives

- Price a run history
- Compute the merge what-if
- Judge a cache by measuring it

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Price the history

Every job that ran is billed per whole minute at its OS rate. The repo is public so the real bill is $0; this is the private-repo equivalent.

In [ ]:
import sys, json, statistics
sys.path.insert(0, str(ROOT / "labs"))
import lab_18_cost_performance_limits as L
from intro_gha.cost import billed_minutes
ran = L.jobs_that_ran()
used, billed = sum(j["sec"] for j in ran), sum(billed_minutes(j["sec"]) for j in ran)
print(len(ran), "jobs |", used, "s used |", billed, "min billed |", round(billed*60/used, 2), "x |", L.cost_mills(ran), "mills")
assert (len(ran), used, billed, L.cost_mills(ran)) == (337, 6357, 390, 2400)

## 2. Why the multiple is so large

Most jobs are tiny, and each is rounded up to a minute.

In [ ]:
short = sum(j["sec"] <= 30 for j in ran)
print(short, "of", len(ran), "jobs are <= 30 s:", round(100*short/len(ran)), "%")
assert short == 316

## 3. The merge what-if

Bill each run's same-OS jobs once, as one merged job.

In [ ]:
merged = L.merged_cost_mills(ran)
print("as run:", L.cost_mills(ran), "| merged:", merged, "| saving:", round(100*(L.cost_mills(ran)-merged)/L.cost_mills(ran)), "%")
assert merged == 1500

## 4. Queue latency

Start time minus creation time, per job.

In [ ]:
qs = sorted(j["queue_s"] for j in ran if j["queue_s"] >= 0)
print("median", statistics.median(qs), "p90", qs[int(.9*len(qs))], "max", qs[-1])
assert statistics.median(qs) == 3

## 5. The cache experiments

pip with a 303 MB cache hit barely helped; uv with a cache was slower than uv without.

In [ ]:
cb = json.loads((ROOT/"fixtures"/"cache_benefit_ch18.json").read_text())
for r in cb["runs"]: print(f'{r["label"]:20} pip install {r["report"]["install_seconds"]:>3} s | job {r["job_seconds"]} s')
for r in cb["uv_runs"]: print(f'{r["label"]:20} uv install  {r["report"]["uv_install_seconds"]:>3} s | job {r["job_seconds"]} s')
uv = {r["label"]: r["job_seconds"] for r in cb["uv_runs"]}
assert uv["uv, cache off"] < uv["uv, cache hit"] < uv["uv, cache miss"]

## Takeaways & Next Steps

- Billed minutes can be several times the time used.
- Merging sub-minute jobs saved 38%.
- Measure before caching; swap the slow tool first.
- Next: Phase 5, Chapter 19 on custom actions.

## Chapter Link

Read: `learning_modules/chapter_18_cost_performance_limits.md`